# Kaggriculture Visible-Price Switch Lab

This small, standalone lab demonstrates a state-conditioned crop decision that uses only the market prices visible in the current observation. It is deliberately separate from the production notebooks: the policy is a teaching example, and the outputs below are deterministic local examples rather than official competition scores.

## The decision

WHEAT and CARROT are fast crops with different yields. For a late planting slot, compare `yield × visible price` for each crop. Switch only when the alternative clears a small margin; otherwise keep the route's original crop. The margin prevents tiny price differences from causing needless seed and action changes.

A real agent must apply the same choice to its `BUY_SEED` order and its later `PLANT` action. It must not use future prices, hidden opponent state, or a post-game reward.

In [ ]:
CROP_YIELD = {"WHEAT": 4.0, "CARROT": 3.0}

def choose_fast_crop(prices, current="CARROT", margin=1.0):
    """Choose a crop from current public prices only."""
    if current not in CROP_YIELD:
        raise ValueError("current must be WHEAT or CARROT")
    values = {crop: CROP_YIELD[crop] * float(prices[crop])
              for crop in CROP_YIELD}
    other = "WHEAT" if current == "CARROT" else "CARROT"
    return other if values[other] - values[current] >= margin else current

examples = [
    {"WHEAT": 4, "CARROT": 5},
    {"WHEAT": 7, "CARROT": 6},
    {"WHEAT": 5, "CARROT": 7},
]
for prices in examples:
    choice = choose_fast_crop(prices, current="CARROT")
    values = {crop: CROP_YIELD[crop] * prices[crop] for crop in CROP_YIELD}
    print(prices, "value=", values, "choice=", choice)

## Why the margin matters

The rule is a guardrail, not a claim that one crop is always best. When the visible values are close, the route keeps its original action and avoids churn. When the gap is clear, the switch is explainable from the current state. This makes the policy easy to audit: every decision can be reproduced from the two prices, the yield table, the current crop, and the chosen margin.

In [ ]:
def route_late_wave(price_trace, current="CARROT", margin=1.0):
    """Return one auditable choice per visible observation."""
    return [choose_fast_crop(prices, current=current, margin=margin)
            for prices in price_trace]

trace = [
    {"WHEAT": 5, "CARROT": 7},
    {"WHEAT": 8, "CARROT": 7},
    {"WHEAT": 6, "CARROT": 8},
]
print(route_late_wave(trace, current="CARROT", margin=1.0))

## Audit checklist

* Inputs are current public prices and the route's current crop.
* The comparison uses fixed, documented yields.
* The same decision must update both seed purchase and planting actions.
* A local example is evidence that the rule runs, not an official Kaggle score.

The useful lesson is the visible-state interface: isolate one decision, state its information boundary, and make its action change reversible and measurable.